# Kenya Spatial Analysis & Computer Vision
**AquaML – Pillar 3**

**Spatial**
- Map water points, density by county and type
- Relate points to rivers / lakes / flood extent (shapefiles in `data/kenya/gis/`)

**Computer Vision**
- Lightweight water-body segmentation demo (same U-Net pattern as notebook 04)
- How to pair Kenya flood / river polygons with satellite tiles for real CV

*Note: Full vector GIS analysis needs `geopandas` + the shapefiles already stored under `data/kenya/gis/`.*


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120


## 1. Water-point spatial overview

In [ ]:
wp = pd.read_csv(ROOT / "data/kenya/kenya_waterpoints_mwater_clean.csv")
print(wp.shape)
print(wp["waterpoint_type"].value_counts())
print("\nTop counties:\n", wp["county"].value_counts().head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# national scatter
types = wp["waterpoint_type"].astype("category")
axes[0].scatter(wp["longitude"], wp["latitude"], c=types.cat.codes, cmap="tab10", s=4, alpha=0.5)
axes[0].set_title("Kenya water points (mWater)")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")

# county counts
vc = wp["county"].value_counts().head(12)
axes[1].barh(vc.index[::-1], vc.values[::-1], color="#1f77b4")
axes[1].set_title("Top counties by water-point count")
axes[1].set_xlabel("Count")
plt.tight_layout(); plt.show()


In [ ]:
# Density heatmap (hexbin)
fig, ax = plt.subplots(figsize=(8, 6))
hb = ax.hexbin(wp["longitude"], wp["latitude"], gridsize=40, cmap="YlOrRd", mincnt=1)
plt.colorbar(hb, ax=ax, label="Point count")
ax.set_title("Water-point density (hexbin)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout(); plt.show()


## 2. GIS layers available on disk

Shapefiles (load with geopandas when installed):

```python
import geopandas as gpd
rivers = gpd.read_file(ROOT / "data/kenya/gis/AFRICOVER_KE_RIVERS.shp")
flood  = gpd.read_file(ROOT / "data/kenya/gis/FloodExtent01.shp")
lakes  = gpd.read_file(ROOT / "data/kenya/gis/KEN_Lakes.shp")
```

Typical workflow once geopandas is available:
1. Reproject all layers to a common CRS (e.g. EPSG:4326 or Arc 1960 UTM)
2. Spatial join water points to nearest river / inside flood polygon
3. Compute distance-to-river and flood exposure features for modelling


In [ ]:
gis_dir = ROOT / "data/kenya/gis"
print("GIS files present:")
for p in sorted(gis_dir.glob("*")):
    if p.suffix.lower() in {".shp", ".dbf", ".prj"}:
        print(" ", p.name, f"({p.stat().st_size // 1024} KB)")


## 3. Computer Vision – water segmentation demo

Same lightweight U-Net approach as the portfolio notebook 04.  
For Kenya production use, replace synthetic patches with:
- Sentinel-2 / Landsat chips around known water points or flood polygons
- Labels from FloodExtent01 or manual digitising


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from src.evaluation import dice_coefficient, iou_score
from src.visualization import plot_segmentation_overlay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def generate_water_patch(size=128, seed=None):
    rng = np.random.default_rng(seed)
    img = rng.normal(80, 25, (size, size, 3)).clip(0, 255).astype(np.uint8)
    img = (img * 0.7 + rng.integers(0, 40, (size, size, 3))).clip(0, 255).astype(np.uint8)
    mask = np.zeros((size, size), dtype=np.uint8)
    for _ in range(rng.integers(1, 4)):
        cy, cx = rng.integers(20, size-20, 2)
        ry, rx = rng.integers(15, 45, 2)
        y, x = np.ogrid[:size, :size]
        blob = ((y - cy)/ry)**2 + ((x - cx)/rx)**2 <= 1
        mask[blob] = 1
        img[blob] = [rng.integers(20, 60), rng.integers(60, 120), rng.integers(140, 220)]
    return img, mask

N = 300
images, masks = [], []
for i in range(N):
    im, m = generate_water_patch(seed=i)
    images.append(im); masks.append(m)
images = np.stack(images); masks = np.stack(masks)
print(images.shape, masks.shape)


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNetTiny(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.down1 = DoubleConv(in_ch, 32); self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(32, 64); self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(64, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2); self.conv2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2); self.conv1 = DoubleConv(64, 32)
        self.out = nn.Conv2d(32, out_ch, 1)
    def forward(self, x):
        d1 = self.down1(x); d2 = self.down2(self.pool1(d1)); bn = self.bottleneck(self.pool2(d2))
        u2 = self.conv2(torch.cat([self.up2(bn), d2], 1))
        u1 = self.conv1(torch.cat([self.up1(u2), d1], 1))
        return torch.sigmoid(self.out(u1))

class WaterDS(Dataset):
    def __init__(self, images, masks):
        self.images = images.astype(np.float32) / 255.0
        self.masks = masks.astype(np.float32)
    def __len__(self): return len(self.images)
    def __getitem__(self, i):
        return (torch.from_numpy(self.images[i]).permute(2,0,1),
                torch.from_numpy(self.masks[i]).unsqueeze(0))

split = int(0.8 * N)
train_loader = DataLoader(WaterDS(images[:split], masks[:split]), batch_size=16, shuffle=True)
test_loader  = DataLoader(WaterDS(images[split:], masks[split:]), batch_size=16)

model = UNetTiny().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCELoss()
model.train()
for epoch in range(12):
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); pred = model(xb); loss = loss_fn(pred, yb)
        loss.backward(); opt.step(); total += loss.item() * len(xb)
    if (epoch + 1) % 4 == 0:
        print(f"Epoch {epoch+1:2d} | loss {total/len(train_loader.dataset):.4f}")


In [ ]:
model.eval()
dice_scores, iou_scores = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        pred = (model(xb.to(device)) > 0.5).cpu().numpy()
        true = yb.numpy()
        for p, t in zip(pred, true):
            dice_scores.append(dice_coefficient(t, p))
            iou_scores.append(iou_score(t, p))
print(f"Mean Dice: {np.mean(dice_scores):.3f} | Mean IoU: {np.mean(iou_scores):.3f}")

xb, yb = next(iter(test_loader))
with torch.no_grad():
    pred = (model(xb.to(device)) > 0.5).cpu().numpy()
for i in range(2):
    img = (xb[i].permute(1,2,0).numpy() * 255).astype(np.uint8)
    plot_segmentation_overlay(img, yb[i,0].numpy(), pred[i,0])
    plt.show()


## Takeaways – Spatial / CV Pillar

- mWater points give immediate national spatial context and density maps.
- Rivers, lakes and flood extent shapefiles are ready for geopandas spatial joins.
- U-Net demo shows the CV training loop; swap in Sentinel-2 chips over FloodExtent01 for a Kenya-specific model.

**Data used**
- `data/kenya/kenya_waterpoints_mwater_clean.csv`
- `data/kenya/gis/` (AFRICOVER rivers, FloodExtent01, KEN_Lakes)
